# 1. **Caricamento dati**
## -Quali variabili sono categoriali e quali numeriche?
Tutte le feature presenti nel dataset, come la longitude, latitude, il numero di stanze, la popolazione e il reddito, sono variabili numeriche. Il target median_house_value rappresenta invece una variabile categoriale, poiché i prezzi originari sono stati discretizzati in 5 classi.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report

In [2]:
csv_path = 'houses_data.csv'
df = pd.read_csv(csv_path)

target = df['median_house_value']
features = df.drop(columns=['median_house_value'])

print("Valori mancanti: ")
print(df.isnull().sum())

Valori mancanti: 
longitude                    0
latitude                     0
housing_median_age           0
total_rooms                  0
total_bedrooms              98
population                   0
households                   0
median_income                0
distance_to_coast            0
distance_to_la               0
distance_to_sandiego         0
distance_to_sanjose          0
distance_to_sanfrancisco     0
median_house_value           0
dtype: int64


# 2. **Visualizzazione dei dati**
## -Ci sono valori mancanti? Quanti e su quali variabili? Come li puoi gestire?
Come si può vedere dall'output della cella precedente, l'unica colonna che presenta valori nulli è total_bedrooms. Per gestirli ho adottato come strategia l'imputazione tramite la media, calcolata esclusivamente sul training set dopo aver effettuato lo split, per prevenire il fenomeno del data leakage.
## -Il target richiede regressione o classificazione? Se classificazione, in quante classi? Le classi sono in numero bilanciato?
Il problema richiede tecniche di classificazione, poiché non si deve prevedere un prezzo esatto continuo, ma una classe di appartenenza. Si le classi sono un numero bilanciato (come si vede dall'output della cella successiva).
## -Esistono correlazioni tra le variabili e il target e tra le variabili stesse? Formula ipotesi e verifica.
Sì, come si può vedere dalla matrice di correlazione, c'è una correlazione tra il median_income e le classi di prezzo più elevate. Anche le variabili geografiche, come la vicinanza alla costa o a San Francisco, mostrano correlazioni con i prezzi

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42 #random_state serve a "mescolare" i dati sempre allo stesso modo
)
print("Distribuzione delle classi nel target:")
print(target.value_counts())
print("\n")

matrice_correlazione = df.corr()

correlazione_target = matrice_correlazione['median_house_value'].sort_values(ascending=False)

print("Correlazione delle feature con la variabile target:")
print(correlazione_target)

Distribuzione delle classi nel target:
median_house_value
3    2112
1    2088
2    2072
4    2034
0    2014
Name: count, dtype: int64


Correlazione delle feature con la variabile target:
median_house_value          1.000000
median_income               0.625336
total_rooms                 0.145598
households                  0.095207
housing_median_age          0.077167
total_bedrooms              0.075369
population                  0.013713
distance_to_sanfrancisco   -0.011040
distance_to_sanjose        -0.024513
longitude                  -0.026161
distance_to_sandiego       -0.127086
distance_to_la             -0.166055
latitude                   -0.183050
distance_to_coast          -0.529245
Name: median_house_value, dtype: float64


# 3. **Codifica della variabile target**
## -Per ogni algoritmo analizzato determina la codifica piu' appropriata della variabile target.
Per i modelli di classificazione lineare binari (Loss Quadratica e Regressione Logistica), ho adottato una codifica One-vs-Rest, allenando 5 classificatori binari indipendenti. Per gli algoritmi k-NN e Alberi Decisionali, non è stata necessaria alcuna codifica speciale, dato che supportano la classificazione multi-classe.

In [4]:
X_train_log = X_train.copy()
X_test_log = X_test.copy()

cols_to_log = ['total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
for col in cols_to_log:
    X_train_log[col] = np.log1p(X_train_log[col])
    X_test_log[col] = np.log1p(X_test_log[col])

# 4. **Cross validation**
## • Imposta una procedura di validazione

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# 5. **Classificazione con loss quadratica**
## -Interpreta i coefficienti ottenuti.
Dall'analisi dei coefficienti emerge chiaramente che le feature dominanti per la classificazione sono il reddito (median_income) e la posizione geografica. Guardando i due estremi dei prezzi, si nota una perfetta coerenza logica del modello:

Per la Classe 4 , il reddito presenta un coefficiente fortemente positivo. Inoltre, le distanze da grandi metropoli come San Francisco e Los Angeles presentano valori negativi: questo indica che una minore distanza aumenta la probabilità di appartenere alla fascia più costosa.
Al contrario, per la Classe 0, il reddito ha un coefficiente fortemente negativo. Questo dimostra che il modello ha imparato ad escludere dalle zone povere i quartieri con un'alta concentrazione di ricchezza."


In [6]:

pipeline_ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')), #per gestire i valori NaN che verrano sostiuiti con la media generale
    ('scaler', StandardScaler()), #rende i valori confrontabili tra loro
    ('classifier', OneVsRestClassifier(RidgeClassifier(alpha=1.0)))
])

cv_scores_ridge = cross_val_score(pipeline_ridge, X_train_log, y_train, cv=cv, scoring='accuracy')
results["1. Loss Quadratica"] = np.mean(cv_scores_ridge)

print(f"Accuratezza CV (Ridge): {np.mean(cv_scores_ridge):.4f}")

pipeline_ridge.fit(X_train_log, y_train)

model_ridge_fitted = pipeline_ridge.named_steps['classifier']

print("\nAnalisi dei Coefficienti per ciascuna classe:")
for i, stimatore in enumerate(model_ridge_fitted.estimators_):
    print(f"\n Modello per la Classe {i}: ")
    coef_classe = pd.Series(stimatore.coef_.flatten(), index=X_train_log.columns)
    print(coef_classe.sort_values(ascending=False).round(4))

Accuratezza CV (Ridge): 0.4921

Analisi dei Coefficienti per ciascuna classe:

 Modello per la Classe 0: 
distance_to_sandiego        1.0856
distance_to_sanfrancisco    0.3566
distance_to_coast           0.3387
total_rooms                 0.2548
longitude                   0.1730
distance_to_la              0.1551
population                  0.1054
housing_median_age          0.0437
households                  0.0425
distance_to_sanjose        -0.1269
median_income              -0.3220
total_bedrooms             -0.4267
latitude                   -0.8092
dtype: float64

 Modello per la Classe 1: 
latitude                    1.3289
longitude                   0.3508
population                  0.2774
distance_to_la              0.1869
distance_to_sanjose         0.0478
total_rooms                 0.0063
total_bedrooms             -0.0055
housing_median_age         -0.0458
median_income              -0.1649
distance_to_coast          -0.1744
distance_to_sanfrancisco   -0.2545
households 

# 6. **Classificazione con loss logistica**
## -Analizza l'effetto del parametro di regolarizzazione.
Modificando il parametro di regolarizzazione, si controlla la penalizzazione sui pesi del modello. Valori molto grandi di regolarizzazione forzano i pesi verso lo zero, rischiando l'underfitting, mentre l'assenza di regolarizzazione può portare il modello a memorizzare il rumore causando overfitting.
## -Quali classi sono più difficili da predire?
Si nota tipicamente che le classi intermedie presentano il maggior tasso di errore. I confini geografici e di reddito per queste case sono molto più sfumati rispetto alle case estremamente economiche o di estremo lusso, causando confusione nel classificatore lineare.


In [7]:
for C_val in [0.01, 1, 100]:
    temp_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', OneVsRestClassifier(LogisticRegression(C=C_val, max_iter=1000, random_state=42)))
    ])
    scores = cross_val_score(temp_pipeline, X_train_log, y_train, cv=cv)
    print(f"Logistic Regression C={C_val} -> Accuratezza CV: {np.mean(scores):.4f}")

pipeline_log_final = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', OneVsRestClassifier(LogisticRegression(C=1, max_iter=1000, random_state=42)))
])

cv_scores_log = cross_val_score(pipeline_log_final, X_train_log, y_train, cv=cv)
results["2. Regressione Logistica"] = np.mean(cv_scores_log)

pipeline_log_final.fit(X_train_log, y_train)

y_pred_log = pipeline_log_final.predict(X_test_log)

print(classification_report(y_test, y_pred_log))

Logistic Regression C=0.01 -> Accuratezza CV: 0.5145
Logistic Regression C=1 -> Accuratezza CV: 0.5469
Logistic Regression C=100 -> Accuratezza CV: 0.5470
              precision    recall  f1-score   support

           0       0.68      0.77      0.72       418
           1       0.52      0.40      0.46       431
           2       0.44      0.41      0.42       411
           3       0.47      0.45      0.46       394
           4       0.67      0.80      0.73       410

    accuracy                           0.57      2064
   macro avg       0.56      0.57      0.56      2064
weighted avg       0.56      0.57      0.56      2064



# 7. **k-NN**
## -Studia l'errore al variare di k valutando se esiste un valore ottimale.
Per K molto bassi, l'accuratezza sul training set è perfetta, ma crolla in validazione (overfitting grave). Aumentando K, il modello generalizza meglio, raggiungendo un picco di accuratezza, per poi peggiorare nuovamente per K troppo elevati, diventando troppo rigido (underfitting).
## -Commenta complessità e accuratezza.
Il k-NN è un algoritmo di lazy learning: il tempo di addestramento è quasi nullo, ma la predizione è computazionalmente costosa poiché richiede il calcolo della distanza tra la nuova casa e tutti i campioni del dataset. Tuttavia, l'accuratezza supera leggermente quella della Regressione Logistica.


In [8]:
k_values = [1, 5, 15, 30]
cv_k_scores = []

for k in k_values:
    pipeline_knn = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    
    scores = cross_val_score(pipeline_knn, X_train_log, y_train, cv=cv, scoring='accuracy')
    cv_k_scores.append(np.mean(scores))
    print(f"k-NN k={k} -> Accuratezza CV: {np.mean(scores)}")

best_k = k_values[np.argmax(cv_k_scores)]
model_knn = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k))
])
model_knn.fit(X_train_log, y_train)
results["3. k-NN"] = max(cv_k_scores)

k-NN k=1 -> Accuratezza CV: 0.5255577000071862
k-NN k=5 -> Accuratezza CV: 0.5517203602483197
k-NN k=15 -> Accuratezza CV: 0.5715856411038581
k-NN k=30 -> Accuratezza CV: 0.5634714744750778



#  8. **Alberi decisionali**
## -Analizza l'effetto dei parametri dell'albero (ad esempio profondità massima, numero minimo di campioni per foglia) sulle prestazioni del modello.
Lasciando l'albero libero di crescere senza vincoli, raggiunge una profondità eccessiva, memorizzando ogni singola anomalia e frammentando lo spazio delle feature in regioni irrilevanti. Impostando un limite alla max_depth e aumentando il min_samples_leaf, si "pota" l'albero, rendendolo più robusto, aumentando significativamente l'accuratezza in Cross-Validation.
## -Quali variabili risultano più importanti secondo il modello? I risultati sono coerenti con l'analisi delle correlazioni?
L'attributo feature_importances_ dell'albero rivela che il nodo radice e i nodi decisionali primari si basano quasi esclusivamente su median_income, latitude e longitude. Questo risultato è perfettamente coerente con quanto emerso dall'analisi delle correlazioni al Punto 2.

In [9]:
from sklearn.pipeline import Pipeline

param_grid = {
    'dt__max_depth': [5, 8, 12, 20, 30, None],
    'dt__min_samples_leaf': [1, 5, 10, 20, 28]
}

pipeline_dt = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('dt', DecisionTreeClassifier(random_state=42))
])

grid_dt = GridSearchCV(pipeline_dt, param_grid, cv=cv, scoring='accuracy')
grid_dt.fit(X_train_log, y_train)

print("Migliori parametri Albero Decisionale:", grid_dt.best_params_)
results["4. Albero Decisionale"] = grid_dt.best_score_

best_pipeline = grid_dt.best_estimator_
best_dt = best_pipeline.named_steps['dt']

importances = best_dt.feature_importances_
feat_imp = pd.Series(importances, index=X_train_log.columns).sort_values(ascending=False)
print("\nImportanza delle variabili:")
print(feat_imp.head(5))

Migliori parametri Albero Decisionale: {'dt__max_depth': 20, 'dt__min_samples_leaf': 10}

Importanza delle variabili:
median_income        0.289509
distance_to_coast    0.244770
longitude            0.071076
distance_to_la       0.065102
latitude             0.060521
dtype: float64


In [10]:
print("TABELLA COMPARATIVA DEI MODELLI\n")
df_results = pd.DataFrame(list(results.items()), columns=['Modello', 'CV Accuracy']).set_index('Modello')
print(df_results.sort_values(by='CV Accuracy', ascending=False))

TABELLA COMPARATIVA DEI MODELLI

                          CV Accuracy
Modello                              
4. Albero Decisionale        0.588663
3. k-NN                      0.571586
2. Regressione Logistica     0.546876
1. Loss Quadratica           0.492128


# 9. **(Opzionale) Un algoritmo a scelta**

# 10. **Conclusioni**
Attraverso la pipeline di Data Science implementata, ho osservato come la corretta gestione dei valori mancanti e delle distribuzioni sbilanciate sia cruciale.
Tra gli algoritmi testati, il modello Albero Decisionale (con i parametri ottimizzati tramite Grid Search) si è rivelato il più performante, riuscendo a isolare geograficamente e per reddito le diverse tipologie abitative e limitando l'overfitting.